# 05. REINVENT4 LibInvent — task-directed RL

**Method**: REINVENT4 LibInvent with task-directed reinforcement learning (RL).

For each parent × task, an RL run uses:
- Fixed Murcko scaffold as the input (attachment-point annotated SMILES)
- A scoring function that rewards candidates achieving exactly the requested
  descriptor delta (e.g. ΔHBA = +1 for the hba_plus1 task)

**Scoring approach**: Built-in REINVENT4 components (`HBondAcceptors`, `HBondDonors`,
`NumRotBond`, `NumAromaticRings`) combined with the `Step` transform.
Step gives score = 1.0 when the descriptor value equals `parent_value + delta`,
and 0.0 otherwise. This matches Scaffold-tuner's exact-delta criterion.

```
task: hba_plus1  (parent HBA = 3)
  → scoring component: HBondAcceptors
  → transform: Step(low=4.0, high=4.0)
  → score = 1.0 if candidate HBA == 4, else 0.0
```

**RL per parent × task**: Each TOML config embeds the parent-specific target value,
so the RL objective is fully customized per parent × task.

**Supplementary control**: `05b_reinvent4_libinvent_prior_sampling.ipynb` provides
a descriptor-blind prior-only sampling baseline for comparison.


In [ ]:
import os, sys, subprocess, traceback, tempfile
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import (
    ACTIVE_PARENT_DATASET, RESULTS_DIR,
    N_CANDIDATES_PER_PARENT, RANDOM_SEED, TASKS, TASK_NAMES,
)
from utils.parents import prepare_parent
from utils.scaffold import scaffold_is_retained
from utils.records import make_candidate_record
from utils.libinvent import make_libinvent_scaffold

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
from configs.benchmark_config import REPO_ROOT
PROJECT_ROOT = REPO_ROOT


## Parameters

In [ ]:
# ── REINVENT4 LibInvent prior ────────────────────────────────────────────────
# Download from Zenodo: https://zenodo.org/records/15641296
# File: libinvent.prior
# Place at the path below before running.
PRIOR_FILE = PROJECT_ROOT / "external/reinvent4_priors/libinvent.prior"

MAX_ATTACHMENT_POINTS = 4   # LibInvent architecture limit (REINVENT4 paper)

# ── RL run parameters ─────────────────────────────────────────────────────────
RL_STEPS       = 50     # RL steps per parent × task
BATCH_SIZE     = 100    # molecules sampled per RL step
# Total generated per parent × task = RL_STEPS × BATCH_SIZE = 5000 raw SMILES
# After filtering, up to N_CANDIDATES_PER_PARENT accepted candidates are kept.

NUM_SMILES     = N_CANDIDATES_PER_PARENT  # max accepted candidates to save
METHOD         = "libinvent"

# ── REINVENT4 component → descriptor mapping ──────────────────────────────────
DESCRIPTOR_TO_REINVENT_COMPONENT = {
    "hbd":  "HBondDonors",
    "hba":  "HBondAcceptors",
    "ar":   "NumAromaticRings",
    "rotb": "NumRotBond",
}

# ── Output files ──────────────────────────────────────────────────────────────
OUTPUT_FILE       = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent.csv"
STATS_FILE        = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent_generation_stats.csv"
INAPPLICABLE_FILE = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent_inapplicable_parents.csv"
ERROR_FILE        = PROJECT_ROOT / f"{RESULTS_DIR}/libinvent_errors.csv"
WORK_DIR          = PROJECT_ROOT / "results/libinvent_rl_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"REINVENT4 version: ", end="")
r = subprocess.run(["reinvent", "-V"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
print(f"PRIOR_FILE: {PRIOR_FILE}  exists={PRIOR_FILE.exists()}")
print(f"RL_STEPS={RL_STEPS}  BATCH_SIZE={BATCH_SIZE}  NUM_SMILES={NUM_SMILES}")


## Step 1: Check / download prior

In [ ]:
PRIOR_FILE.parent.mkdir(parents=True, exist_ok=True)
if not PRIOR_FILE.exists():
    zenodo_url = "https://zenodo.org/records/15641296/files/libinvent.prior?download=1"
    print(f"Prior not found. Attempting download from:\n  {zenodo_url}")
    r = subprocess.run(
        ["curl", "-L", "--fail", "-o", str(PRIOR_FILE), zenodo_url],
        capture_output=True, text=True,
    )
    if r.returncode != 0 or not PRIOR_FILE.exists():
        raise FileNotFoundError(
            f"Could not download LibInvent prior.\n"
            f"Please download manually from https://zenodo.org/records/15641296\n"
            f"and place at: {PRIOR_FILE}\n"
            f"curl stderr: {r.stderr[:300]}"
        )
    print(f"Downloaded: {PRIOR_FILE.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Prior already present: {PRIOR_FILE}  ({PRIOR_FILE.stat().st_size / 1e6:.1f} MB)")


## Step 2: Prepare LibInvent scaffold SMILES

Each parent's Murcko scaffold is annotated with attachment points (`[*:0]`, `[*:1]`, ...).
Parents where no valid scaffold can be built, or with > 4 attachment points
(beyond LibInvent's architecture limit), are recorded as inapplicable
and treated as 0-candidate entries in Success@k (→ False).


In [ ]:
df = pd.read_csv(ACTIVE_PARENT_DATASET)
if "canonical_smiles" not in df.columns and "smiles" in df.columns:
    df = df.rename(columns={"smiles": "canonical_smiles"})
if "parent_id" not in df.columns and "chembl_id" in df.columns:
    df = df.rename(columns={"chembl_id": "parent_id"})
print(f"Parents loaded: {len(df)}")

scaffold_map = {}   # parent_id → (parent_smiles, libinvent_scaffold_smiles, parent_desc)
inapplicable = []

for _, row in df.iterrows():
    prepared = prepare_parent(row)
    if prepared is None:
        inapplicable.append({"parent_id": row["parent_id"], "reason": "prepare_parent_failed"})
        continue
    parent_id, parent_smiles, parent_desc, _ = prepared
    libinv_smi, reason = make_libinvent_scaffold(parent_smiles)
    if libinv_smi is None:
        inapplicable.append({"parent_id": parent_id, "reason": reason})
        continue
    mol = Chem.MolFromSmiles(libinv_smi)
    n_ap = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 0)
    if n_ap > MAX_ATTACHMENT_POINTS:
        inapplicable.append({"parent_id": parent_id, "reason": "too_many_attachment_points"})
        continue
    scaffold_map[parent_id] = (parent_smiles, libinv_smi, parent_desc)

print(f"Applicable parents   : {len(scaffold_map)}")
print(f"Inapplicable parents : {len(inapplicable)}")
pd.DataFrame(inapplicable).to_csv(INAPPLICABLE_FILE, index=False)


## Step 3: TOML config generator

For each parent × task, a TOML config is generated with:
- `HBondAcceptors` / `HBondDonors` / `NumRotBond` / `NumAromaticRings` as the
  scoring component (selected by task)
- `Step` transform with `low = high = parent_value + delta`
  → score = 1.0 iff candidate descriptor == target value

The `Step` transform gives score 1.0 when `low ≤ descriptor_value ≤ high`,
and 0.0 otherwise. Setting `low = high = target` implements exact-delta scoring,
matching Scaffold-tuner's validation criterion.

**parent_value is embedded per TOML**: each config file is specific to one parent,
so there is no shared `_PARENT_VALUE` variable.


In [ ]:
def write_rl_config(task_name, scaffold_smi, parent_desc, work_dir, prior_path):
    """Write a REINVENT4 staged_learning TOML for one parent × task.

    The scoring function uses the appropriate RDKit descriptor component
    with a Step transform tuned to parent_value + delta.

    Parameters
    ----------
    task_name   : str   e.g. "hba_plus1"
    scaffold_smi: str   LibInvent scaffold SMILES with attachment points
    parent_desc : dict  parent descriptor values {"hbd":..., "hba":..., ...}
    work_dir    : Path  directory for config and output files
    prior_path  : Path  LibInvent prior .prior file

    Returns
    -------
    config_path : Path  written TOML file
    raw_csv     : Path  where REINVENT4 will write generated SMILES
    smi_path    : Path  scaffold SMILES input file
    """
    from configs.benchmark_config import TASKS
    task_cfg   = TASKS[task_name]
    descriptor = task_cfg["descriptor"]     # e.g. "hba"
    delta      = task_cfg["delta"]          # +1 or -1
    component  = DESCRIPTOR_TO_REINVENT_COMPONENT[descriptor]  # e.g. "HBondAcceptors"

    # Parent-specific target value
    parent_value = parent_desc[descriptor]
    target_value = float(parent_value + delta)

    # Sanitize task_name and parent_id for filenames
    slug = task_name  # e.g. "hba_plus1"

    smi_path   = work_dir / f"scaffold_{slug}.smi"
    raw_csv    = work_dir / f"raw_{slug}.csv"
    agent_ckpt = work_dir / f"agent_{slug}.prior"
    config_path = work_dir / f"config_{slug}.toml"

    # Write scaffold SMILES input (single scaffold per run)
    smi_path.write_text(scaffold_smi + "\n")

    toml = f"""
run_type = "staged_learning"
device = "cpu"
tb_logdir = ""

[parameters]
prior_file = "{prior_path}"
agent_file = "{prior_path}"
smiles_file = "{smi_path}"
summary_csv_prefix = "{work_dir / slug}"
batch_size = {BATCH_SIZE}
randomize_smiles = true

[[stage]]
max_steps = {RL_STEPS}
max_score = 1.0
chkpt_file = "{agent_ckpt}"
termination = "simple"
min_steps = 10

[stage.scoring]
type = "geometric_mean"

[[stage.scoring.component]]
[stage.scoring.component.{component}]
name = "{descriptor}_target"
weight = 1.0

[[stage.scoring.component.{component}.endpoint]]
name = "{descriptor}_step"
weight = 1.0

[stage.scoring.component.{component}.endpoint.transform]
type = "Step"
low = {target_value}
high = {target_value}
""".strip()

    config_path.write_text(toml + "\n")
    return config_path, raw_csv, smi_path


## Step 4: Smoke test — 1 parent × 1 task

Before running all 1000 × 7 jobs, verify that:
1. REINVENT4 starts and loads the prior correctly
2. Scaffold SMILES is accepted as input
3. Scoring function is called (Step transform with correct target)
4. RL produces valid SMILES
5. Candidates pass scaffold retention check


In [ ]:
# Pick the first applicable parent for smoke test
test_pid, (test_smi, test_scaffold_smi, test_desc) = next(iter(scaffold_map.items()))
test_task = "hba_plus1"
test_target = test_desc["hba"] + 1

print(f"Smoke test parent  : {test_pid}")
print(f"Parent SMILES      : {test_smi}")
print(f"LibInvent scaffold : {test_scaffold_smi}")
print(f"Task               : {test_task}  (target HBA = {test_target})")
print()

smoke_work = WORK_DIR / "smoke_test"
smoke_work.mkdir(parents=True, exist_ok=True)
smoke_log = smoke_work / "smoke.log"

cfg_path, raw_csv, smi_path = write_rl_config(
    test_task, test_scaffold_smi, test_desc, smoke_work, PRIOR_FILE
)
print(f"Config written → {cfg_path}")
print()
print("=== TOML content ===")
print(cfg_path.read_text())


In [ ]:
print("Running REINVENT4 smoke test...")
env = os.environ.copy()

result = subprocess.run(
    ["reinvent", "-l", str(smoke_log), "--seed", str(RANDOM_SEED), str(cfg_path)],
    capture_output=True, text=True, env=env, timeout=300,
)

print(f"Return code: {result.returncode}")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("REINVENT4 smoke test failed — check TOML format and prior file.")
else:
    print("REINVENT4 smoke test completed successfully.")

# Parse smoke test output
if raw_csv.exists():
    df_smoke = pd.read_csv(raw_csv)
    print(f"Raw output: {len(df_smoke)} rows  columns={list(df_smoke.columns)}")
else:
    # Try summary CSV
    summary_csvs = list(smoke_work.glob("*.csv"))
    print(f"Output files: {summary_csvs}")
    if summary_csvs:
        df_smoke = pd.read_csv(summary_csvs[0])
        print(df_smoke.head())


In [ ]:
# Diagnostic: check how many smoke test candidates have HBA == target
from rdkit.Chem import rdMolDescriptors

SMILES_COL = next((c for c in df_smoke.columns if "smiles" in c.lower()), None)
if SMILES_COL:
    hba_counts = []
    for smi in df_smoke[SMILES_COL].dropna():
        mol = Chem.MolFromSmiles(smi)
        if mol:
            hba_counts.append(rdMolDescriptors.CalcNumHBA(mol))
    if hba_counts:
        import numpy as np
        print(f"Smoke test HBA distribution (target={test_target}):")
        values, counts = np.unique(hba_counts, return_counts=True)
        for v, c in zip(values, counts):
            marker = " ← TARGET" if v == test_target else ""
            print(f"  HBA={v}: {c} candidates{marker}")
        success_rate = sum(1 for h in hba_counts if h == test_target) / len(hba_counts)
        print(f"Exact delta success rate: {success_rate:.1%}")


## Step 5: Full run — all parents × 7 tasks

In [ ]:
records, gen_stats, error_rows = [], [], []

for task_name in tqdm(TASK_NAMES, desc="tasks"):
    task_cfg   = TASKS[task_name]
    descriptor = task_cfg["descriptor"]
    delta      = task_cfg["delta"]

    for parent_id, (parent_smiles, libinv_smi, parent_desc) in tqdm(
        scaffold_map.items(), desc=task_name, leave=False
    ):
        parent_mol = Chem.MolFromSmiles(parent_smiles)
        task_work  = WORK_DIR / task_name / parent_id
        task_work.mkdir(parents=True, exist_ok=True)
        log_path   = task_work / "run.log"

        try:
            cfg_path, raw_csv, smi_path = write_rl_config(
                task_name, libinv_smi, parent_desc, task_work, PRIOR_FILE
            )
            result = subprocess.run(
                ["reinvent", "-l", str(log_path), "--seed", str(RANDOM_SEED), str(cfg_path)],
                capture_output=True, text=True, env=env, timeout=300,
            )
            if result.returncode != 0:
                raise RuntimeError(result.stderr[-500:])
        except Exception as e:
            error_rows.append({
                "parent_id": parent_id, "task": task_name,
                "error_type": type(e).__name__, "error_message": str(e)[:500],
            })
            gen_stats.append({
                "method": METHOD, "task": task_name, "parent_id": parent_id,
                "n_examined": 0, "n_invalid": 0, "n_scaffold_failed": 0,
                "n_duplicate": 0, "n_valid_unique": 0, "rl_error": True,
            })
            continue

        # Collect raw SMILES from summary CSV
        # REINVENT4 staged_learning writes summary CSV named {summary_csv_prefix}_*.csv
        output_csvs = list(task_work.glob("*.csv"))
        if not output_csvs:
            error_rows.append({
                "parent_id": parent_id, "task": task_name,
                "error_type": "no_output", "error_message": f"no CSV in {task_work}",
            })
            continue

        df_raw = pd.read_csv(sorted(output_csvs)[-1])  # latest CSV
        SMILES_COL = next((c for c in df_raw.columns if "smiles" in c.lower()), None)
        if SMILES_COL is None:
            error_rows.append({
                "parent_id": parent_id, "task": task_name,
                "error_type": "no_smiles_col", "error_message": str(df_raw.columns.tolist()),
            })
            continue

        # candidate_rank definition for LibInvent RL:
        # REINVENT4 staged_learning CSV rows are written in RL generation order
        # (step 1 batch, step 2 batch, ...). The CSV is NOT sorted by score.
        # Therefore, row order = RL generation time order (same meaning as
        # Random / CReM where row order = generation attempt order).
        # We collect raw_smis in row order, then filter invalid / scaffold_failed
        # / duplicate entries. The rank of accepted candidates reflects their
        # position in this RL generation time sequence — no post-hoc sorting.
        raw_smis = df_raw[SMILES_COL].dropna().tolist()

        # Filter: invalid → scaffold_failed → duplicate → accepted
        n_examined = len(raw_smis)
        n_invalid=0; n_scaffold_failed=0; n_duplicate=0
        candidates=[]; seen=set([parent_smiles])

        # Process ALL raw SMILES for generation statistics.
        # Accepted candidates are collected in full; only top-NUM_SMILES are saved.
        for smi in raw_smis:
            cand_mol = Chem.MolFromSmiles(smi)
            if cand_mol is None:
                n_invalid += 1; continue
            canon = Chem.MolToSmiles(cand_mol)
            if not scaffold_is_retained(parent_mol, cand_mol):
                n_scaffold_failed += 1; continue
            if canon in seen:
                n_duplicate += 1; continue
            seen.add(canon); candidates.append(canon)

        n_valid_unique = len(candidates)
        assert n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique == n_examined, (
            f"{parent_id}/{task_name}: stats mismatch"
        )
        # Keep at most NUM_SMILES candidates for the benchmark CSV.
        candidates = candidates[:NUM_SMILES]

        gen_stats.append({
            "method": METHOD, "task": task_name, "parent_id": parent_id,
            "n_examined": n_examined, "n_invalid": n_invalid,
            "n_scaffold_failed": n_scaffold_failed, "n_duplicate": n_duplicate,
            "n_valid_unique": n_valid_unique, "rl_error": False,
        })

        for rank, cand_smi in enumerate(candidates, start=1):
            rec = make_candidate_record(
                method=METHOD, parent_id=parent_id,
                parent_smiles=parent_smiles, task_name=task_name,
                candidate_rank=rank, candidate_smiles=cand_smi,
                parent_desc=parent_desc,
            )
            if rec is not None:
                records.append(rec)

print(f"Total records: {len(records)}")
print(f"Errors: {len(error_rows)}")


In [ ]:
pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
pd.DataFrame(gen_stats).to_csv(STATS_FILE, index=False)
pd.DataFrame(error_rows).to_csv(ERROR_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
print(f"Saved → {STATS_FILE}")
print(f"Saved → {ERROR_FILE} ({len(error_rows)} errors)")
